In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


class ResidualBlock3d(nn.Module):
    """Residual block for better feature refinement."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.act = nn.GELU()

    def forward(self, x):
        residual = x
        x = self.act(self.conv1(x))
        x = self.conv2(x)
        return self.act(x + residual)  # Skip connection


class DeepDecoderBlock(nn.Module):
    """
    Deep decoder block with learned upsampling + multiple refinement layers.
    Much stronger than single conv refinement.
    """
    def __init__(self, channels, num_residual_blocks=2):
        super().__init__()
        
        # Learned upsampling
        self.conv_tp = nn.ConvTranspose3d(
            channels, channels, 
            kernel_size=3, 
            stride=(2, 2, 1), 
            padding=1,
            output_padding=(1, 1, 0)
        )
        
        # Multiple residual blocks for refinement
        self.residual_blocks = nn.ModuleList([
            ResidualBlock3d(channels) for _ in range(num_residual_blocks)
        ])
        
        # Final refinement
        self.final_conv = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.act = nn.GELU()

    def forward(self, x, target_x, target_y):
        # Learned upsampling
        x = self.conv_tp(x)
        
        # Crop/pad to exact target size
        curr_x, curr_y = x.shape[2], x.shape[3]
        if curr_x != target_x or curr_y != target_y:
            if curr_x >= target_x and curr_y >= target_y:
                x = x[:, :, :target_x, :target_y, :]
            else:
                pad_x = max(0, target_x - curr_x)
                pad_y = max(0, target_y - curr_y)
                x = F.pad(x, (0, 0, 0, pad_y, 0, pad_x))
        
        # Deep refinement with residual blocks
        for res_block in self.residual_blocks:
            x = res_block(x)
        
        # Final refinement
        x = self.act(self.final_conv(x))
        
        return x

class DeepDynamicUNetDecoder3d(nn.Module):
    """
    Deeper decoder with residual refinement blocks.
    
    Total depth per upsampling stage:
    - 1 ConvTranspose3d (upsampling)
    - 2 ResidualBlocks = 4 Conv3d layers
    - 1 final Conv3d
    = 6 learnable layers per block
    
    With 4 blocks: 24 total layers (vs 8 in original)
    """
    def __init__(self, channels, num_layers=4, num_residual_blocks=2):
        super().__init__()
        self.num_layers = num_layers
        self.layers = nn.ModuleList([
            DeepDecoderBlock(channels, num_residual_blocks) 
            for _ in range(num_layers)
        ])

    def forward(self, x, final_nx, final_ny):
        curr_x, curr_y = x.shape[2], x.shape[3]
        
        # Calculate intermediate target sizes
        targets_x = []
        targets_y = []
        
        for i in range(self.num_layers):
            if i < self.num_layers - 1:
                targets_x.append(min(curr_x * (2 ** (i + 1)), final_nx))
                targets_y.append(min(curr_y * (2 ** (i + 1)), final_ny))
            else:
                targets_x.append(final_nx)
                targets_y.append(final_ny)
        
        # Apply decoder layers with deep refinement
        for i, layer in enumerate(self.layers):
            x = layer(x, targets_x[i], targets_y[i])
        
        return x


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


class FlexibleResidualBlock3d(nn.Module):
    """
    Flexible residual block that allows any number of convolution layers.
    
    You can control:
    - Number of conv layers (num_convs) - default 2
    - Kernel size (default 3)
    - Activation function (default GELU)
    
    Input / Output shape: [nb, channels, x, y, t] → same shape
    """
    def __init__(
        self,
        channels,
        num_convs=2,           # ← Flexible: how many conv layers (2, 3, 4, ...)
        kernel_size=3,         # Can be 3, 5, 7, etc.
        activation=nn.GELU,    # Can be GELU, ReLU, LeakyReLU, etc.
        use_bn=True            # Whether to use BatchNorm after each conv
    ):
        super().__init__()
        
        # Store parameters
        self.num_convs = num_convs
        self.act = activation()
        
        # Create list of conv + optional BatchNorm layers
        self.convs = nn.ModuleList()
        for i in range(num_convs):
            conv = nn.Conv3d(channels, channels, kernel_size=kernel_size, padding=kernel_size//2)
            self.convs.append(conv)
            if use_bn:
                self.convs.append(nn.BatchNorm3d(channels))

    def forward(self, x):
        """
        Forward pass with flexible number of convolutions.
        
        - Applies each conv + activation sequentially
        - Adds residual (skip connection) at the end
        """
        residual = x  # Save original input for skip connection
        
        # Apply flexible chain of conv + activation (+ BatchNorm)
        for layer in self.convs:
            x = layer(x)
            # Apply activation only after conv layers (skip BatchNorm)
            if isinstance(layer, nn.Conv3d):
                x = self.act(x)
        
        # Add residual connection and final activation
        return self.act(x + residual)


# ------------------------------------------------------------------
# Updated DeepDecoderBlock using the flexible residual block
# ------------------------------------------------------------------
class DeepDecoderBlock(nn.Module):
    """
    Deep decoder block with learned upsampling + flexible deep refinement.
    
    Now uses FlexibleResidualBlock3d so you can control conv depth easily.
    """
    def __init__(
        self,
        channels,
        num_residual_blocks=2,
        convs_per_residual=2,     # ← NEW: control convs per residual block
        kernel_size=3
    ):
        super().__init__()
        
        # Learned upsampling
        self.conv_tp = nn.ConvTranspose3d(
            channels, channels,
            kernel_size=3,
            stride=(2, 2, 1),
            padding=1,
            output_padding=(1, 1, 0)
        )
        
        # Flexible residual blocks
        self.residual_blocks = nn.ModuleList([
            FlexibleResidualBlock3d(
                channels,
                num_convs=convs_per_residual,
                kernel_size=kernel_size
            )
            for _ in range(num_residual_blocks)
        ])
        
        # Final refinement conv
        self.final_conv = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.act = nn.GELU()

    def forward(self, x, target_x, target_y):
        # Learned upsampling
        x = self.conv_tp(x)
        
        # Crop/pad to exact target size
        curr_x, curr_y = x.shape[2], x.shape[3]
        if curr_x != target_x or curr_y != target_y:
            if curr_x >= target_x and curr_y >= target_y:
                x = x[:, :, :target_x, :target_y, :]
            else:
                pad_x = max(0, target_x - curr_x)
                pad_y = max(0, target_y - curr_y)
                x = F.pad(x, (0, 0, 0, pad_y, 0, pad_x))
        
        # Deep refinement with flexible residual blocks
        for res_block in self.residual_blocks:
            x = res_block(x)
        
        # Final refinement
        x = self.act(self.final_conv(x))
        
        return x


# ------------------------------------------------------------------
# Full Deep Dynamic UNet Decoder (unchanged structure, now uses flexible block)
# ------------------------------------------------------------------
class DeepDynamicUNetDecoder3d(nn.Module):
    """
    Complete deep decoder using flexible residual blocks.
    
    Now you can control:
    - Number of upsampling stages (num_layers)
    - Number of residual blocks per stage
    - Number of convs inside each residual block
    """
    def __init__(
        self,
        channels,
        num_layers=4,
        num_residual_blocks=2,
        convs_per_residual=2,     # ← NEW: control conv depth in each res block
        kernel_size=3
    ):
        super().__init__()
        self.num_layers = num_layers
        
        self.layers = nn.ModuleList([
            DeepDecoderBlock(
                channels,
                num_residual_blocks=num_residual_blocks,
                convs_per_residual=convs_per_residual,
                kernel_size=kernel_size
            )
            for _ in range(num_layers)
        ])

    def forward(self, x, final_nx, final_ny):
        curr_x, curr_y = x.shape[2], x.shape[3]
        
        targets_x = []
        targets_y = []
        for i in range(self.num_layers):
            if i < self.num_layers - 1:
                next_x = min(curr_x * (2 ** (i + 1)), final_nx)
                next_y = min(curr_y * (2 ** (i + 1)), final_ny)
            else:
                next_x = final_nx
                next_y = final_ny
            targets_x.append(next_x)
            targets_y.append(next_y)
        
        for i, layer in enumerate(self.layers):
            x = layer(x, targets_x[i], targets_y[i])
        
        return x



In [ ]:
# ------------------------------------------------------------------
# Quick test to show flexibility and shapes (FIXED)
# ------------------------------------------------------------------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Define the model parameters explicitly here
    channels = 32
    num_layers = 4                  # Number of upsampling stages
    num_residual_blocks = 3         # Residual blocks per stage
    convs_per_residual = 3          # Convolutions per residual block

    # Example input sizes
    nb = 2
    coarse_x, coarse_y = 100, 50      # Coarse input spatial size
    nt = 15                         # Time steps
    final_nx, final_ny = 200, 200     # Target fine spatial size (×8 upscaling)

    # Create dummy coarse input
    x = torch.randn(nb, channels, coarse_x, coarse_y, nt, device=device)

    print("\nInput shape:")
    print(f"  x: {x.shape}  [nb, channels, coarse_x, coarse_y, nt]")

    # Create the model with the chosen parameters
    model = DeepDynamicUNetDecoder3d(
        channels=channels,
        num_layers=num_layers,
        num_residual_blocks=num_residual_blocks,
        convs_per_residual=convs_per_residual,
        kernel_size=3
    ).to(device)

    # Run forward pass
    output = model(x, final_nx, final_ny)

    print("\nOutput shape:")
    print(f"  output: {output.shape}  [nb, channels, final_nx, final_ny, nt]")



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResidualBlock3d(nn.Module):
    """
    Basic 3D residual block for deep feature refinement.
    
    - Two 3x3x3 convolutions + GELU activation
    - Residual (skip) connection for stable training
    - Input/Output shape: [nb, channels, x, y, t] → same
    """
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.act = nn.GELU()

    def forward(self, x):
        residual = x
        x = self.act(self.conv1(x))
        x = self.conv2(x)
        return self.act(x + residual)


class DeepRefinementBlock(nn.Module):
    """
    One refinement stage: multiple residual blocks + final conv.
    
    Used repeatedly to progressively refine the features.
    No upsampling here — model expects already fine-sized input.
    """
    def __init__(self, channels, num_residual_blocks=3):
        super().__init__()
        # Stack of residual blocks for depth
        self.res_blocks = nn.ModuleList([
            ResidualBlock3d(channels) for _ in range(num_residual_blocks)
        ])
        # Final conv to polish the output
        self.final_conv = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.act = nn.GELU()

    def forward(self, x):
        # Deep residual refinement
        for block in self.res_blocks:
            x = block(x)
        # Final touch-up
        x = self.act(self.final_conv(x))
        return x


class MagnifierModel(nn.Module):
    """
    Final magnifier model that matches dummy_magnifier shape exactly.
    
    INPUT:
        patch_input: [nb, C_in, P_fine, P_fine, nt]
                     Typically C_in = 2 (channel 0: interpolated coarse u,
                                         channel 1: fine bed repeated over time)
                     P_fine = N * f (fine patch size)
    
    OUTPUT:
        [nb, 1, P_fine, P_fine, nt]
        Refined high-resolution u patch (single channel)
    
    Key features:
    - No internal upsampling → expects already fine-sized input
    - Progressive refinement with multiple deep blocks
    - Keeps exact input spatial/temporal size
    - Very flexible depth (change num_refinement_blocks and num_residual_per_block)
    """
    def __init__(
        self,
        in_channels=2,                  # Usually 2: coarse u interp + fine bed
        base_channels=32,               # Internal feature dimension
        num_refinement_blocks=4,        # Number of refinement stages
        num_residual_per_block=3        # Residual blocks per stage
    ):
        super().__init__()
        
        # Initial lifting: map input channels to internal features
        self.lifting = nn.Conv3d(in_channels, base_channels, kernel_size=3, padding=1)
        
        # Stack of deep refinement blocks
        self.refinement_stages = nn.ModuleList([
            DeepRefinementBlock(
                base_channels,
                num_residual_blocks=num_residual_per_block
            )
            for _ in range(num_refinement_blocks)
        ])
        
        # Final projection to 1 output channel (refined u)
        self.final_projection = nn.Conv3d(base_channels, 1, kernel_size=1)

    def forward(self, patch_input):
        """
        Forward pass: refine the input patch to high-res u.
        
        Args:
            patch_input: [nb, C_in, P_fine, P_fine, nt]
        
        Returns:
            refined_u: [nb, 1, P_fine, P_fine, nt]
        """
        # Step 1: Lift input to higher-dimensional features
        x = self.lifting(patch_input)          # [nb, base_channels, P_fine, P_fine, nt]
        
        # Step 2: Progressive deep refinement
        for stage in self.refinement_stages:
            x = stage(x)
        
        # Step 3: Project back to single channel (refined u)
        refined_u = self.final_projection(x)   # [nb, 1, P_fine, P_fine, nt]
        
        return refined_u



In [ ]:

# ------------------------------------------------------------------
# Quick test to verify exact dummy-like input/output shapes
# ------------------------------------------------------------------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Testing MagnifierModel on device: {device}")

    # Typical real-world shapes
    nb = 2               # batch size
    N = 5                # coarse patch size
    f = 8                # upscaling factor
    P_fine = N * f       # 40
    nt = 15              # time steps
    C_in = 2             # channels: coarse u interp + fine bed

    # Create dummy patch input
    patch_input = torch.randn(nb, C_in, P_fine, P_fine, nt, device=device)

    print("\nInput shape (matches dummy_magnifier):")
    print(f"  patch_input: {patch_input.shape}  [nb, {C_in}, {P_fine}, {P_fine}, nt]")

    # Create the model
    model = MagnifierModel(
        in_channels=C_in,
        base_channels=32,
        num_refinement_blocks=4,         # 4 refinement stages
        num_residual_per_block=3         # 3 residual blocks per stage
    ).to(device)

    # Run forward pass
    output = model(patch_input)

    print("\nOutput shape (exactly matches dummy_magnifier):")
    print(f"  output: {output.shape}  [nb, 1, {P_fine}, {P_fine}, nt]")

    print("\nMagnifier model ready! Input/output shapes are identical to dummy function.")
    print(" - Input: [nb, 2, P_fine, P_fine, nt]")
    print(" - Output: [nb, 1, P_fine, P_fine, nt]")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# final decoder version
# ------------------------------------------------------------------
# Your EXACT unchanged SpectralConv2d
# ------------------------------------------------------------------
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super(SpectralConv2d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = (1 / (in_channels * out_channels))
        self.weights1_real = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2))
        self.weights1_imag = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2))
        self.weights2_real = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2))
        self.weights2_imag = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2))

    def compl_mul2d(self, input_real, input_imag, weights_real, weights_imag):
        return torch.einsum("bixy,ioxy->boxy", input_real, weights_real) - \
               torch.einsum("bixy,ioxy->boxy", input_imag, weights_imag), \
               torch.einsum("bixy,ioxy->boxy", input_real, weights_imag) + \
               torch.einsum("bixy,ioxy->boxy", input_imag, weights_real)

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfft2(x)
        x_ft_real, x_ft_imag = x_ft.real, x_ft.imag

        out_ft_real = torch.zeros(batchsize, self.out_channels, x.size(-2), x.size(-1)//2 + 1, device=x.device)
        out_ft_imag = torch.zeros_like(out_ft_real)

        out_ft_real[:, :, :self.modes1, :self.modes2], out_ft_imag[:, :, :self.modes1, :self.modes2] = \
            self.compl_mul2d(x_ft_real[:, :, :self.modes1, :self.modes2], x_ft_imag[:, :, :self.modes1, :self.modes2],
                             self.weights1_real, self.weights1_imag)

        out_ft_real[:, :, -self.modes1:, :self.modes2], out_ft_imag[:, :, -self.modes1:, :self.modes2] = \
            self.compl_mul2d(x_ft_real[:, :, -self.modes1:, :self.modes2], x_ft_imag[:, :, -self.modes1:, :self.modes2],
                             self.weights2_real, self.weights2_imag)

        out_ft = torch.complex(out_ft_real, out_ft_imag)
        x = torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)))
        return x


# ------------------------------------------------------------------
# FNOBlock2d: Now correctly handles 5D input
# ------------------------------------------------------------------
class FNOBlock2d(nn.Module):
    def __init__(self, channels, modes_x, modes_y):
        super().__init__()
        self.spectral = SpectralConv2d(channels, channels, modes_x, modes_y)
        self.local = nn.Conv3d(channels, channels, kernel_size=1)
        self.activation = nn.GELU()

    def forward(self, x):
        """
        Input: [nb, c, height, width, t]
        - Reshape t to batch dim for spectral conv (expects 4D)
        - Apply 2D spectral on spatial (x,y)
        - Reshape back to 5D
        - Add local 3D conv + activation
        """
        nb, c, height, width, t = x.shape

        # Reshape: treat time as batch → [nb*t, c, height, width]
        x_reshaped = x.permute(0, 4, 1, 2, 3).contiguous().view(nb * t, c, height, width)

        # Apply pure 2D spectral convolution
        x_spec = self.spectral(x_reshaped)

        # Reshape back to original 5D shape [nb, c, height, width, t]
        x_spec = x_spec.view(nb, t, c, height, width).permute(0, 2, 3, 4, 1)

        # Local conv on original 5D tensor
        x_local = self.local(x)

        # Combine and activate
        return self.activation(x_spec + x_local)


# ------------------------------------------------------------------
# ResidualBlock3d (unchanged)
# ------------------------------------------------------------------
class ResidualBlock3d(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.act = nn.GELU()

    def forward(self, x):
        residual = x
        x = self.act(self.conv1(x))
        x = self.conv2(x)
        return self.act(x + residual)


# ------------------------------------------------------------------
# Final MagnifierModel: Pure 2D FNO + Deep CNN
# ------------------------------------------------------------------
class MagnifierModel(nn.Module):
    """
    Pure 2D FNO Magnifier Model (spatial only + deep CNN refinement)
    
    INPUT:   [nb, 2, P_fine, P_fine, nt]
    OUTPUT:  [nb, 1, P_fine, P_fine, nt]
    
    - FNO only on spatial dimensions (x,y) → no time FFT
    - FNOBlock2d handles 5D input correctly
    - Exact match to dummy_magnifier shapes
    """
    def __init__(
        self,
        in_channels=2,
        base_channels=48,
        num_fno_blocks=3,
        fno_modes_x=16,
        fno_modes_y=16,
        num_refinement_blocks=4,
        num_residual_per_block=3
    ):
        super().__init__()
        
        self.lifting = nn.Conv3d(in_channels, base_channels, kernel_size=3, padding=1)
        
        self.fno_mixer = nn.ModuleList([
            FNOBlock2d(base_channels, fno_modes_x, fno_modes_y)
            for _ in range(num_fno_blocks)
        ])
        
        self.refinement_stages = nn.ModuleList([
            nn.Sequential(*[
                ResidualBlock3d(base_channels)
                for _ in range(num_residual_per_block)
            ])
            for _ in range(num_refinement_blocks)
        ])
        
        self.final_projection = nn.Conv3d(base_channels, 1, kernel_size=1)

    def forward(self, patch_input):
        x = self.lifting(patch_input)
        
        for fno_block in self.fno_mixer:
            x = fno_block(x)
        
        for stage in self.refinement_stages:
            x = stage(x)
        
        refined_u = self.final_projection(x)
        return refined_u


# ------------------------------------------------------------------
# Quick test (exact dummy shapes)
# ------------------------------------------------------------------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Testing Pure 2D FNO MagnifierModel on device: {device}")

    nb = 2
    P_fine = 20
    nt = 88
    C_in = 2

    patch_input = torch.randn(nb, C_in, P_fine, P_fine, nt, device=device)

    print("\nInput shape:")
    print(f"  patch_input: {patch_input.shape}  [nb, {C_in}, {P_fine}, {P_fine}, nt]")

    model = MagnifierModel(
        in_channels=C_in,
        base_channels=48,
        num_fno_blocks=3,
        fno_modes_x=16,
        fno_modes_y=16,
        num_refinement_blocks=4,
        num_residual_per_block=3
    ).to(device)

    output = model(patch_input)

    print("\nOutput shape:")
    print(f"  output: {output.shape}  [nb, 1, {P_fine}, {P_fine}, nt]")

    print("\nModel ready!")
    print(" - Pure 2D FNO (spatial only)")
    print(" - Correct 5D handling")
    print(" - Exact match to dummy_magnifier")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


# ------------------------------------------------------------------
# SpectralConv2d (UNCHANGED - Your exact implementation)
# ------------------------------------------------------------------
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super(SpectralConv2d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = (1 / (in_channels * out_channels))
        self.weights1_real = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2))
        self.weights1_imag = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2))
        self.weights2_real = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2))
        self.weights2_imag = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2))

    def compl_mul2d(self, input_real, input_imag, weights_real, weights_imag):
        return torch.einsum("bixy,ioxy->boxy", input_real, weights_real) - \
               torch.einsum("bixy,ioxy->boxy", input_imag, weights_imag), \
               torch.einsum("bixy,ioxy->boxy", input_real, weights_imag) + \
               torch.einsum("bixy,ioxy->boxy", input_imag, weights_real)

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfft2(x)
        x_ft_real, x_ft_imag = x_ft.real, x_ft.imag

        out_ft_real = torch.zeros(batchsize, self.out_channels, x.size(-2), x.size(-1)//2 + 1, device=x.device)
        out_ft_imag = torch.zeros_like(out_ft_real)

        out_ft_real[:, :, :self.modes1, :self.modes2], out_ft_imag[:, :, :self.modes1, :self.modes2] = \
            self.compl_mul2d(x_ft_real[:, :, :self.modes1, :self.modes2], x_ft_imag[:, :, :self.modes1, :self.modes2],
                             self.weights1_real, self.weights1_imag)

        out_ft_real[:, :, -self.modes1:, :self.modes2], out_ft_imag[:, :, -self.modes1:, :self.modes2] = \
            self.compl_mul2d(x_ft_real[:, :, -self.modes1:, :self.modes2], x_ft_imag[:, :, -self.modes1:, :self.modes2],
                             self.weights2_real, self.weights2_imag)

        out_ft = torch.complex(out_ft_real, out_ft_imag)
        x = torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)))
        return x


# ------------------------------------------------------------------
# FNOBlock2d with Dropout (2D spatial FFT only)
# ------------------------------------------------------------------
class FNOBlock2d(nn.Module):
    def __init__(self, channels, modes_x, modes_y, dropout=0.0):
        super().__init__()
        self.spectral = SpectralConv2d(channels, channels, modes_x, modes_y)
        self.local = nn.Conv2d(channels, channels, kernel_size=1)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        # Input shape: [nb, c, height, width, t]
        nb, c, height, width, t = x.shape

        # Reshape to treat time as batch: [nb*t, c, height, width]
        x_reshaped = x.permute(0, 4, 1, 2, 3).reshape(nb * t, c, height, width)

        # Apply pure 2D spectral conv (FFT on height/width only)
        x_spec = self.spectral(x_reshaped)
        
        # Apply local 2D conv
        x_local = self.local(x_reshaped)

        # Combine and activate
        x_combined = self.activation(x_spec + x_local)
        x_combined = self.dropout(x_combined)

        # Reshape back to 5D [nb, c, height, width, t]
        return x_combined.reshape(nb, t, c, height, width).permute(0, 2, 3, 4, 1)


# ------------------------------------------------------------------
# Residual Block with Dropout and Channel Expansion
# ------------------------------------------------------------------
class ResidualBlock3d(nn.Module):
    def __init__(self, in_channels, out_channels=None, dropout=0.0):
        super().__init__()
        out_channels = out_channels or in_channels
        self.expand = in_channels != out_channels
        
        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1)
        self.act = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        
        # Projection for residual if channels change
        if self.expand:
            self.residual_proj = nn.Conv3d(in_channels, out_channels, kernel_size=1)
        else:
            self.residual_proj = nn.Identity()

    def forward(self, x):
        residual = self.residual_proj(x)
        x = self.act(self.conv1(x))
        x = self.dropout(x)
        x = self.conv2(x)
        return self.act(x + residual)


# ------------------------------------------------------------------
# Spatial Attention Module
# ------------------------------------------------------------------
class SpatialAttention3d(nn.Module):
    """Lightweight spatial attention to focus on important regions"""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv3d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        max_pool = torch.max(x, dim=1, keepdim=True)[0]
        avg_pool = torch.mean(x, dim=1, keepdim=True)
        attention_input = torch.cat([max_pool, avg_pool], dim=1)
        attention_map = self.sigmoid(self.conv(attention_input))
        return x * attention_map


# ------------------------------------------------------------------
# Multi-Scale Pyramid Pooling Module
# ------------------------------------------------------------------
class PyramidPooling3d(nn.Module):
    """Multi-scale context aggregation"""
    def __init__(self, in_channels, pool_sizes=[2, 4, 8]):
        super().__init__()
        self.pool_sizes = pool_sizes
        out_channels_per_pool = in_channels // len(pool_sizes)
        
        self.convs = nn.ModuleList([
            nn.Conv3d(in_channels, out_channels_per_pool, kernel_size=1)
            for _ in pool_sizes
        ])
        
        # Fusion: in_channels (original) + out_channels_per_pool * len(pool_sizes) (pooled)
        total_channels = in_channels + out_channels_per_pool * len(pool_sizes)
        self.fusion = nn.Conv3d(total_channels, in_channels, kernel_size=1)

    def forward(self, x):
        _, _, h, w, t = x.shape
        pyramid_features = [x]
        
        for pool_size, conv in zip(self.pool_sizes, self.convs):
            pooled = F.adaptive_avg_pool3d(x, (h // pool_size, w // pool_size, t // pool_size))
            pooled = conv(pooled)
            upsampled = F.interpolate(pooled, size=(h, w, t), mode='trilinear', align_corners=False)
            pyramid_features.append(upsampled)
        
        return self.fusion(torch.cat(pyramid_features, dim=1))


# ------------------------------------------------------------------
# Enhanced Magnifier Model with Progressive Channel EXPANSION
# ------------------------------------------------------------------
class MagnifierModel(nn.Module):
    """
    Enhanced 2D FNO Magnifier Model for Spatial Upscaling with:
    - 2D spatial FFT only (no temporal overhead)
    - Progressive channel EXPANSION (correct for upscaling)
    - Multi-scale pyramid pooling
    - Spatial attention
    - Skip connections between stages
    - Dropout regularization
    - Gradient checkpointing support
    
    INPUT:   [nb, 2, P_fine, P_fine, nt]
    OUTPUT:  [nb, 1, P_fine, P_fine, nt]
    """
    def __init__(
        self,
        in_channels=2,
        base_channels=48,           # Starting channels
        num_fno_blocks=5,           # Increased for better mixing
        fno_modes_x=16,             # Higher spatial modes
        fno_modes_y=16,
        num_refinement_blocks=5,    # Progressive expansion stages
        num_residual_per_block=3,
        channel_multipliers=[1.0, 1.33, 1.67, 2.0, 2.0],  # Progressive expansion
        dropout=0.1,
        use_attention=True,
        use_pyramid_pooling=True,
        use_gradient_checkpointing=False
    ):
        super().__init__()
        
        assert len(channel_multipliers) == num_refinement_blocks, \
            f"Need {num_refinement_blocks} multipliers, got {len(channel_multipliers)}"
        
        self.use_gradient_checkpointing = use_gradient_checkpointing
        
        # Initial lifting
        self.lifting = nn.Conv3d(in_channels, base_channels, kernel_size=3, padding=1)
        
        # FNO mixer blocks with dropout (2D spatial FFT)
        self.fno_mixer = nn.ModuleList([
            FNOBlock2d(base_channels, fno_modes_x, fno_modes_y, dropout=dropout)
            for _ in range(num_fno_blocks)
        ])
        
        # Multi-scale pyramid pooling after FNO
        self.pyramid_pooling = PyramidPooling3d(base_channels) if use_pyramid_pooling else nn.Identity()
        
        # Spatial attention after FNO
        self.attention = SpatialAttention3d(base_channels) if use_attention else nn.Identity()
        
        # Progressive channel expansion in refinement stages
        self.refinement_stages = nn.ModuleList()
        current_channels = base_channels
        self.channel_counts = [base_channels]
        
        for i in range(num_refinement_blocks):
            next_channels = int(base_channels * channel_multipliers[i])
            
            stage_blocks = []
            for j in range(num_residual_per_block):
                if j == 0:
                    # First block in stage handles channel expansion
                    stage_blocks.append(ResidualBlock3d(current_channels, next_channels, dropout=dropout))
                    current_channels = next_channels
                else:
                    # Subsequent blocks maintain channels
                    stage_blocks.append(ResidualBlock3d(current_channels, current_channels, dropout=dropout))
            
            self.refinement_stages.append(nn.Sequential(*stage_blocks))
            self.channel_counts.append(current_channels)
        
        # Skip connection fusion layers (from FNO output to each refinement stage)
        self.skip_fusions = nn.ModuleList([
            nn.Conv3d(base_channels + ch, ch, kernel_size=1)
            for ch in self.channel_counts[1:]  # Skip first (no fusion needed)
        ])
        
        # Final projection from expanded channels to output
        self.final_projection = nn.Conv3d(current_channels, 1, kernel_size=1)
        
        # Initialize weights
        self._initialize_weights()

    def _initialize_weights(self):
        """Proper weight initialization"""
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Conv3d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def _checkpoint_forward(self, module, x):
        """Conditional gradient checkpointing"""
        if self.use_gradient_checkpointing and self.training:
            return checkpoint(module, x, use_reentrant=False)
        else:
            return module(x)

    def forward(self, patch_input):
        # Lifting
        x = self.lifting(patch_input)
        
        # FNO mixer with checkpointing (2D spatial FFT only)
        for fno_block in self.fno_mixer:
            x = self._checkpoint_forward(fno_block, x)
        
        # Multi-scale context
        x = self.pyramid_pooling(x)
        
        # Spatial attention
        x = self.attention(x)
        
        # Store FNO output for skip connections
        fno_output = x
        
        # Progressive refinement with channel expansion and skip connections
        for i, stage in enumerate(self.refinement_stages):
            x = self._checkpoint_forward(stage, x)
            
            # Add skip connection from FNO output
            if i < len(self.skip_fusions):
                x = self.skip_fusions[i](torch.cat([x, fno_output], dim=1))
        
        # Final projection to single channel output
        refined_u = self.final_projection(x)
        
        return refined_u


# ------------------------------------------------------------------
# Multi-Scale Loss for Training
# ------------------------------------------------------------------
class MultiScaleLoss(nn.Module):
    """
    Multi-scale loss that supervises at different resolutions
    Helps with training stability and detail preservation
    """
    def __init__(self, scales=[1, 2, 4], weights=[1.0, 0.5, 0.25]):
        super().__init__()
        self.scales = scales
        self.weights = weights
        self.base_loss = nn.MSELoss()
    
    def forward(self, pred, target):
        total_loss = 0.0
        
        for scale, weight in zip(self.scales, self.weights):
            if scale == 1:
                loss = self.base_loss(pred, target)
            else:
                # Downsample both pred and target
                pred_down = F.avg_pool3d(pred, kernel_size=scale, stride=scale)
                target_down = F.avg_pool3d(target, kernel_size=scale, stride=scale)
                loss = self.base_loss(pred_down, target_down)
            
            total_loss += weight * loss
        
        return total_loss


# ------------------------------------------------------------------
# Usage Examples and Testing
# ------------------------------------------------------------------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Testing Enhanced 2D FNO Magnifier on device: {device}\n")
    print("=" * 70)

    # ------------------------------------------------------------------
    # Example 1: Enhanced Configuration with Progressive Expansion
    # ------------------------------------------------------------------
    print("\n[Example 1] Enhanced Model with Progressive Channel Expansion")
    print("-" * 70)

    # Typical real-world shapes
    nb = 64               # batch size
    N = 5                # coarse patch size
    f = 8                # upscaling factor
    P_fine = N * f       # 40
    nt = 88              # time steps
    C_in = 2             # channels: coarse u interp + fine bed

    # Create dummy patch input
    patch_input = torch.randn(nb, C_in, P_fine, P_fine, nt, device=device)



    model_enhanced = MagnifierModel(
        in_channels=C_in,
        base_channels=32,
        num_fno_blocks=5,
        fno_modes_x=16,
        fno_modes_y=16,
        num_refinement_blocks=5,
        num_residual_per_block=3,
        channel_multipliers=[1.0, 1.33, 1.67, 2.0, 2.0],  # 48→64→80→96→96
        dropout=0.1,
        use_attention=True,
        use_pyramid_pooling=True,
        use_gradient_checkpointing=False
    ).to(device)
    
    output = model_enhanced(patch_input)
    
    print(f"Input shape:  {patch_input.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Channel progression: {model_enhanced.channel_counts}")
    
    total_params = sum(p.numel() for p in model_enhanced.parameters())
    print(f"Total parameters: {total_params:,}")


In [ ]:

    # ------------------------------------------------------------------
    # Example 2: Original 2D Configuration (for comparison)
    # ------------------------------------------------------------------
    print("\n[Example 2] Original 2D Configuration")
    print("-" * 70)
    
    model_original = MagnifierModel(
        in_channels=C_in,
        base_channels=48,
        num_fno_blocks=3,
        fno_modes_x=16,
        fno_modes_y=16,
        num_refinement_blocks=4,
        num_residual_per_block=3,
        channel_multipliers=[1.0, 1.0, 1.0, 1.0],  # Constant 48 channels
        dropout=0.0,
        use_attention=False,
        use_pyramid_pooling=False
    ).to(device)
    
    output_orig = model_original(patch_input)
    
    print(f"Input shape:  {patch_input.shape}")
    print(f"Output shape: {output_orig.shape}")
    print(f"Channel progression: {model_original.channel_counts}")
    
    total_params_orig = sum(p.numel() for p in model_original.parameters())
    print(f"Total parameters: {total_params_orig:,}")
    print(f"Parameter increase: {(total_params / total_params_orig - 1) * 100:.1f}%")

    # ------------------------------------------------------------------
    # Example 3: Training with Multi-Scale Loss
    # ------------------------------------------------------------------
    print("\n[Example 3] Training Loop with Multi-Scale Loss")
    print("-" * 70)
    
    target = torch.randn(nb, 1, P_fine, P_fine, nt, device=device)
    
    criterion = MultiScaleLoss(scales=[1, 2, 4], weights=[1.0, 0.5, 0.25])
    optimizer = torch.optim.AdamW(model_enhanced.parameters(), lr=1e-4, weight_decay=1e-5)
    
    model_enhanced.train()
    optimizer.zero_grad()
    
    pred = model_enhanced(patch_input)
    loss = criterion(pred, target)
    
    loss.backward()
    optimizer.step()
    
    print(f"Prediction shape: {pred.shape}")
    print(f"Multi-scale loss: {loss.item():.6f}")
    print("Training step successful!")

    # ------------------------------------------------------------------
    # Example 4: Different Channel Expansion Strategies
    # ------------------------------------------------------------------
    print("\n[Example 4] Different Channel Expansion Strategies")
    print("-" * 70)
    
    strategies = {
        "Aggressive (48→128)": [1.0, 1.5, 2.0, 2.5, 2.67],
        "Moderate (48→96)": [1.0, 1.33, 1.67, 2.0, 2.0],
        "Conservative (48→72)": [1.0, 1.17, 1.33, 1.5, 1.5],
        "Constant (48→48)": [1.0, 1.0, 1.0, 1.0, 1.0]
    }
    
    for name, multipliers in strategies.items():
        config = MagnifierModel(
            in_channels=C_in,
            base_channels=48,
            num_fno_blocks=5,
            fno_modes_x=16,
            fno_modes_y=16,
            num_refinement_blocks=5,
            channel_multipliers=multipliers,
            dropout=0.1,
            use_attention=True,
            use_pyramid_pooling=True
        ).to(device)
        
        params = sum(p.numel() for p in config.parameters())
        print(f"{name:25s}: {config.channel_counts} → {params:,} params")

    # ------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------
    print("\n" + "=" * 70)
    print("All tests passed! Model is ready for spatial upscaling.")
    print("\nKey Features for Upscaling:")
    print("  ✓ 2D spatial FFT only (no temporal overhead)")
    print("  ✓ Progressive channel EXPANSION (48→96 by default)")
    print("  ✓ Multi-scale pyramid pooling for context")
    print("  ✓ Spatial attention for adaptive weighting")
    print("  ✓ Skip connections from FNO to refinement")
    print("  ✓ Dropout regularization")
    print("  ✓ Gradient checkpointing support")
    print("  ✓ Multi-scale loss for better training")
    print("=" * 70)

# encoder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class SpectralConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_x, modes_y, modes_t):
        super(SpectralConv3d, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes_x #Number of Fourier modes to multiply, at most floor(N/2) + 1
        self.modes2 = modes_y
        self.modes3 = modes_t

        self.scale = (1 / (in_channels * out_channels))
        # Initialize real and imaginary parts separately
        self.weights1_real = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3))
        self.weights1_imag = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3))
        
        self.weights2_real = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3))
        self.weights2_imag = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3))
        
        self.weights3_real = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3))
        self.weights3_imag = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3))
        
        self.weights4_real = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3))
        self.weights4_imag = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, self.modes3))

    def compl_mul3d(self, input_real, input_imag, weights_real, weights_imag):
        # (batch, in_channel, x,y,t), (in_channel, out_channel, x,y,t) -> (batch, out_channel, x,y,t)
        return torch.einsum("bixyz,ioxyz->boxyz", input_real, weights_real) - \
               torch.einsum("bixyz,ioxyz->boxyz", input_imag, weights_imag), \
               torch.einsum("bixyz,ioxyz->boxyz", input_real, weights_imag) + \
               torch.einsum("bixyz,ioxyz->boxyz", input_imag, weights_real)

    def forward(self, x):
        batchsize = x.shape[0]
        
        # Compute Fourier coefficients up to factor of e^(- something constant)
        x_ft = torch.fft.rfftn(x, dim=[-3,-2,-1])
        x_ft_real, x_ft_imag = x_ft.real, x_ft.imag
        
        # Multiply relevant Fourier modes
        out_ft_real = torch.zeros(batchsize, self.out_channels, x.size(-3), x.size(-2), x.size(-1)//2 + 1, device=x.device)
        out_ft_imag = torch.zeros(batchsize, self.out_channels, x.size(-3), x.size(-2), x.size(-1)//2 + 1, device=x.device)
        
        # First set of modes
        out_ft_real[:, :, :self.modes1, :self.modes2, :self.modes3], \
        out_ft_imag[:, :, :self.modes1, :self.modes2, :self.modes3] = \
            self.compl_mul3d(x_ft_real[:, :, :self.modes1, :self.modes2, :self.modes3],
                           x_ft_imag[:, :, :self.modes1, :self.modes2, :self.modes3],
                           self.weights1_real, self.weights1_imag)
        
        # Second set of modes
        out_ft_real[:, :, -self.modes1:, :self.modes2, :self.modes3], \
        out_ft_imag[:, :, -self.modes1:, :self.modes2, :self.modes3] = \
            self.compl_mul3d(x_ft_real[:, :, -self.modes1:, :self.modes2, :self.modes3],
                           x_ft_imag[:, :, -self.modes1:, :self.modes2, :self.modes3],
                           self.weights2_real, self.weights2_imag)
        
        # Third set of modes
        out_ft_real[:, :, :self.modes1, -self.modes2:, :self.modes3], \
        out_ft_imag[:, :, :self.modes1, -self.modes2:, :self.modes3] = \
            self.compl_mul3d(x_ft_real[:, :, :self.modes1, -self.modes2:, :self.modes3],
                           x_ft_imag[:, :, :self.modes1, -self.modes2:, :self.modes3],
                           self.weights3_real, self.weights3_imag)
        
        # Fourth set of modes
        out_ft_real[:, :, -self.modes1:, -self.modes2:, :self.modes3], \
        out_ft_imag[:, :, -self.modes1:, -self.modes2:, :self.modes3] = \
            self.compl_mul3d(x_ft_real[:, :, -self.modes1:, -self.modes2:, :self.modes3],
                           x_ft_imag[:, :, -self.modes1:, -self.modes2:, :self.modes3],
                           self.weights4_real, self.weights4_imag)
        
        # Combine real and imaginary parts
        out_ft = torch.complex(out_ft_real, out_ft_imag)
        
        # Return to physical space
        x = torch.fft.irfftn(out_ft, s=(x.size(-3), x.size(-2), x.size(-1)))
        return x



class UNetEncoder3d(nn.Module):
    def __init__(self, channels, num_layers, target_mx, target_my):
        """
        Flexible 3D Encoder that downsamples spatially to specific dimensions.
        
        Args:
            channels (int): Constant number of channels (C).
            num_layers (int): Number of convolutional downsampling steps.
            target_mx (int): Exact target size for dimension x.
            target_my (int): Exact target size for dimension y.
        """
        super(UNetEncoder3d, self).__init__()
        
        layers = []
        for i in range(num_layers):
            layers.append(
                nn.Sequential(
                    nn.Conv3d(channels, channels, kernel_size=3, 
                              stride=(2, 2, 1), padding=1),
                    nn.GELU(),
                    nn.Conv3d(channels, channels, kernel_size=3, 
                              stride=1, padding=1),
                    nn.GELU()
                )
            )
            
        self.encoder = nn.Sequential(*layers)
        
        # This layer forces the output to be exactly (target_mx, target_my, nt)
        # It handles the math if (nx / 2^layers) doesn't perfectly match mx/my.
        self.final_pool = nn.AdaptiveAvgPool3d((target_mx, target_my, None))

    def forward(self, x):
        # x shape: [batch, C, nx, ny, nt]
        x = self.encoder(x)
        # final_pool preserves the last dimension (nt) automatically if passed None
        # but to be safe with all PyTorch versions, we pass the current nt.
        nt = x.shape[-1]
        x = nn.functional.adaptive_avg_pool3d(x, (self.final_pool.output_size[0], 
                                                 self.final_pool.output_size[1], 
                                                 nt))
        return x


class ResidualBlock3d(nn.Module):
    """Residual block for better feature refinement."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.act = nn.GELU()

    def forward(self, x):
        residual = x
        x = self.act(self.conv1(x))
        x = self.conv2(x)
        return self.act(x + residual)  # Skip connection


class DeepDecoderBlock(nn.Module):
    """
    Deep decoder block with learned upsampling + multiple refinement layers.
    Much stronger than single conv refinement.
    """
    def __init__(self, channels, num_residual_blocks=2):
        super().__init__()
        
        # Learned upsampling
        self.conv_tp = nn.ConvTranspose3d(
            channels, channels, 
            kernel_size=3, 
            stride=(2, 2, 1), 
            padding=1,
            output_padding=(1, 1, 0)
        )
        
        # Multiple residual blocks for refinement
        self.residual_blocks = nn.ModuleList([
            ResidualBlock3d(channels) for _ in range(num_residual_blocks)
        ])
        
        # Final refinement
        self.final_conv = nn.Conv3d(channels, channels, kernel_size=3, padding=1)
        self.act = nn.GELU()

    def forward(self, x, target_x, target_y):
        # Learned upsampling
        x = self.conv_tp(x)
        
        # Crop/pad to exact target size
        curr_x, curr_y = x.shape[2], x.shape[3]
        if curr_x != target_x or curr_y != target_y:
            if curr_x >= target_x and curr_y >= target_y:
                x = x[:, :, :target_x, :target_y, :]
            else:
                pad_x = max(0, target_x - curr_x)
                pad_y = max(0, target_y - curr_y)
                x = F.pad(x, (0, 0, 0, pad_y, 0, pad_x))
        
        # Deep refinement with residual blocks
        for res_block in self.residual_blocks:
            x = res_block(x)
        
        # Final refinement
        x = self.act(self.final_conv(x))
        
        return x


class DeepDynamicUNetDecoder3d(nn.Module):
    """
    Deeper decoder with residual refinement blocks.
    
    Total depth per upsampling stage:
    - 1 ConvTranspose3d (upsampling)
    - 2 ResidualBlocks = 4 Conv3d layers
    - 1 final Conv3d
    = 6 learnable layers per block
    
    With 4 blocks: 24 total layers (vs 8 in original)
    """
    def __init__(self, channels, num_layers=4, num_residual_blocks=2):
        super().__init__()
        self.num_layers = num_layers
        self.layers = nn.ModuleList([
            DeepDecoderBlock(channels, num_residual_blocks) 
            for _ in range(num_layers)
        ])

    def forward(self, x, final_nx, final_ny):
        curr_x, curr_y = x.shape[2], x.shape[3]
        
        # Calculate intermediate target sizes
        targets_x = []
        targets_y = []
        
        for i in range(self.num_layers):
            if i < self.num_layers - 1:
                targets_x.append(min(curr_x * (2 ** (i + 1)), final_nx))
                targets_y.append(min(curr_y * (2 ** (i + 1)), final_ny))
            else:
                targets_x.append(final_nx)
                targets_y.append(final_ny)
        
        # Apply decoder layers with deep refinement
        for i, layer in enumerate(self.layers):
            x = layer(x, targets_x[i], targets_y[i])
        
        return x


class FNO3d(nn.Module):
    def __init__(self, T_in, T_out, modes_x, modes_y, modes_t, width=20, encoder_kernel_size_x=128, encoder_kernel_size_y=128, encoder_num_layers=4):
        super(FNO3d, self).__init__()

        """
        The overall network. It contains 4 layers of the Fourier layer.
        1. Lift the input to the desire channel dimension by self.fc0 .
        2. 4 layers of the integral operators u' = (W + K)(u).
            W defined by self.w; K defined by self.conv .
        3. Project from the channel space to the output space by self.fc1 and self.fc2 .
        
        input: the solution of the first 10 timesteps + 3 locations (u(1, x, y), ..., u(10, x, y),  x, y, t). It's a constant function in time, except for the last index.
        input shape: (batchsize, x=64, y=64, t=40, c=13)
        output: the solution of the next 40 timesteps
        output shape: (batchsize, x=64, y=64, t=40, c=1)
        """
        self.T_in = T_in
        self.T_out = T_out
        self.modes1 = modes_x
        self.modes2 = modes_y
        self.modes3 = modes_t
        self.encoder_kernel_size_x = encoder_kernel_size_x
        self.encoder_kernel_size_y = encoder_kernel_size_y
        self.encoder_num_layers = encoder_num_layers
        self.width = width
        self.padding = 6 # pad the domain if input is non-periodic
        self.encoder = UNetEncoder3d(channels=self.width, 
                                            num_layers=self.encoder_num_layers, 
                                            target_mx=self.encoder_kernel_size_x, 
                                            target_my=self.encoder_kernel_size_y)

        # # Initialize here instead of None
        # self.decoder = DeepDynamicUNetDecoder3d(
        #                                     channels=self.width,           # Your width
        #                                     num_layers=4,          # 4 upsampling stages (20→40→80→160→313)
        #                                     num_residual_blocks=1  # 2 residual blocks per stage
# )
        self.fc0 = nn.Linear(self.T_in + 5, self.width)
        # input channel is 12: the solution of the first 10 timesteps + 3 locations (u(1, x, y), ..., u(10, x, y),  x, y, t)

        self.conv0 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv1 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv2 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        self.conv3 = SpectralConv3d(self.width, self.width, self.modes1, self.modes2, self.modes3)
        
        self.w0 = nn.Conv3d(self.width, self.width, 1)
        self.w1 = nn.Conv3d(self.width, self.width, 1)
        self.w2 = nn.Conv3d(self.width, self.width, 1)
        self.w3 = nn.Conv3d(self.width, self.width, 1)
        
        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, forcing, u0, B):
        original_nx, original_ny = u0.shape[1], u0.shape[2]
        u0 = u0.unsqueeze(-2).repeat(1, 1, 1, self.T_out + 1, 1)
        forcing = forcing.unsqueeze(-1)
        grid = self.get_grid(u0.shape, u0.device)
        
        B = B.unsqueeze(-1).unsqueeze(-1).repeat(1, 1, 1, self.T_out + 1, 1)
        # torch.Size([2, 64, 64, 40, 10]) torch.Size([2, 64, 64, 40, 3])
        
        x = torch.cat((forcing, u0, grid, B), dim=-1)  # shape [nb, nx, ny, T_out, C]
        x = self.fc0(x)                      # shape [nb, nx, ny, T_out, width]
        x = x.permute(0, 4, 1, 2, 3)         # shape [nb, width, nx, ny, T_out]
        
        if self.padding > 0:
            x = F.pad(x, [0,self.padding])  # pad the domain if input is non-periodic shape [B, C, W, H, T+self.padding]
        
        x = self.encoder(x)                   # shape [nb, width, mx, my, T_out]
        
        x1 = self.conv0(x)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x2 = self.w3(x)
        x = x1 + x2                        # shape [B, C, W, H, T+self.padding]

        x = x[..., :-self.padding]          # shape [B, C, W, H, T]
        # x = self.decoder(x, original_nx, original_ny)                # shape [B, C, nx, ny, T]
        x = x.permute(0, 2, 3, 4, 1)        # shape [B, W, H, T, C]  
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x).squeeze(-1)
        return x[..., 1:]


    def get_grid(self, shape, device):
        """
        Generate normalized coordinate grids for 3D data.
        
        Args:
            shape: (batchsize, size_x, size_y, size_z)
            device: torch device
        
        Returns:
            grid: [batchsize, size_x, size_y, size_z, 3] containing (x, y, t) coordinates
        """
        batchsize, size_x, size_y, size_z = shape[0], shape[1], shape[2], shape[3]
        
        # Create 1D coordinate arrays
        x = torch.linspace(0, 1, size_x, device=device)
        y = torch.linspace(0, 1, size_y, device=device)
        t = torch.linspace(0, 1, size_z, device=device)
        
        # Create 2D meshgrid for x and y
        x_grid, y_grid = torch.meshgrid(x, y, indexing='ij')  # [size_x, size_y]
        
        # Expand to full shape [batchsize, size_x, size_y, size_z, 1]
        x_grid = x_grid[None, :, :, None, None].expand(batchsize, -1, -1, size_z, 1)
        y_grid = y_grid[None, :, :, None, None].expand(batchsize, -1, -1, size_z, 1)
        t_grid = t[None, None, None, :, None].expand(batchsize, size_x, size_y, -1, 1)
        
        # Concatenate along last dimension
        return torch.cat([x_grid, y_grid, t_grid], dim=-1)


In [ ]:
if __name__ == "__main__":
    device = torch.device(f"cuda")  # Set device based on rank
    
    T_in=1
    T_out=88
    model = FNO3d(T_in=T_in, T_out=T_out, 
                modes_x=12, modes_y=12, modes_t=12, 
                width=40,
                encoder_kernel_size_x=82,
                encoder_kernel_size_y=41,
                encoder_num_layers=4)
    model = model.to(device)  # Move model to the correct GPU before wrapping with DDP
    nbatch, s0, s1 = 2, 328, 164
    u_in = torch.rand(nbatch, s0, s1, T_in).to(device)  # Move input tensors to the same device
    forcing = torch.rand(nbatch, s0, s1, T_in + T_out).to(device)
    parameters = torch.rand(nbatch, s0, s1).to(device)

    u_out = model(forcing, u_in, parameters)   # u_in:  [nb, nx, ny,  T_in] parameters: [nb, nx, ny]
    print(f"{u_out.shape}")           # u_out: [nb, nx, ny,  T_out] 



In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LightMagnifier(nn.Module):
    def __init__(self, in_channels=3, width=32, num_refinement_layers=3):
        """
        Args:
            in_channels (int): Now 3 (Interpolated, Bed, Bathtub).
            width (int): Hidden dimension (model capacity).
            num_refinement_layers (int): How many layers in the bottleneck.
        """
        super().__init__()
        self.in_channels = in_channels
        
        # 1. Unified Feature Extraction (More efficient than separate branches)
        # We process all input channels together to learn spatial correlations immediately
        self.feature_extractor = nn.Sequential(
            nn.Conv3d(in_channels, width, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        # 2. Flexible Refinement Bottleneck
        # Allows you to make the model deeper or shallower based on PSU cluster limits
        refinement_list = []
        for _ in range(num_refinement_layers):
            refinement_list.append(nn.Conv3d(width, width, kernel_size=3, padding=1))
            refinement_list.append(nn.GELU())
        self.refinement = nn.Sequential(*refinement_list)
        
        # 3. Final Projection to Residual
        self.projection = nn.Conv3d(width, 1, kernel_size=1)

    def forward(self, x):
        """
        x shape: [nb, in_channels, Pf, Pf, nt]
        """
        # --- RESIDUAL ANCHOR SELECTION ---
        # If we have 3 channels, the "Bathtub" (channel 2) is our best physical guess.
        # If we only have 2, the "Interpolated" (channel 0) is our guess.
        if self.in_channels == 3:
            physical_baseline = x[:, 2:3, :, :, :] # Channel 2: Bathtub
        else:
            physical_baseline = x[:, 0:1, :, :, :] # Channel 0: Interpolated Coarse
        
        # 1. Extract Features from all augmented inputs
        feat = self.feature_extractor(x)
        
        # 2. Deep Refinement
        feat = self.refinement(feat)
        
        # 3. Predict the Correction (Residual)
        residual = self.projection(feat)
        
        # 4. FINAL SUM: Physics + Neural Correction
        return physical_baseline + residual

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpatialCrossAttention(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.width = width
        # Query: Derived from Water (Coarse + Bathtub)
        self.q_conv = nn.Conv3d(width, width, kernel_size=1)
        # Key: Derived from Terrain (Bed)
        self.k_conv = nn.Conv3d(width, width, kernel_size=1)
        # Value: Derived from Terrain (Bed)
        self.v_conv = nn.Conv3d(width, width, kernel_size=1)
        
        self.softmax = nn.Softmax(dim=-1)
        self.out_conv = nn.Conv3d(width, width, kernel_size=1)

    def forward(self, water_feat, bed_feat):
        # Shapes: [nb, c, h, w, t]
        b, c, h, w, t = water_feat.shape
        
        # We treat each time step as a separate spatial relationship
        # Reshape to [b*t, c, h*w] for efficient matrix multiplication
        q = self.q_conv(water_feat).permute(0, 4, 1, 2, 3).reshape(-1, c, h*w)
        k = self.k_conv(bed_feat).permute(0, 4, 1, 2, 3).reshape(-1, c, h*w)
        v = self.v_conv(bed_feat).permute(0, 4, 1, 2, 3).reshape(-1, c, h*w)

        # 1. Compute Attention Map (Relate water to terrain)
        # energy: [b*t, h*w, h*w]
        energy = torch.bmm(q.transpose(1, 2), k) 
        attention = self.softmax(energy / (c ** 0.5))

        # 2. Apply Attention to Terrain Values
        # out: [b*t, c, h*w]
        out = torch.bmm(v, attention.transpose(1, 2))
        
        # 3. Reshape back to original 5D tensor
        out = out.reshape(b, t, c, h, w).permute(0, 2, 3, 4, 1)
        return self.out_conv(out)

class CrossAttentionMagnifier(nn.Module):
    def __init__(self, width=32, num_refinement_layers=3):
        super().__init__()
        
        # Branch 1: Terrain Feature Extractor (Channel 1)
        self.bed_net = nn.Sequential(
            nn.Conv3d(1, width, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        # Branch 2: Water Feature Extractor (Channel 0 and 2)
        self.water_net = nn.Sequential(
            nn.Conv3d(2, width, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        # The Cross-Attention Engine
        self.cross_attn = SpatialCrossAttention(width)
        
        # Post-Attention Refinement
        layers = []
        for _ in range(num_refinement_layers):
            layers.append(nn.Conv3d(width, width, kernel_size=3, padding=1))
            layers.append(nn.GELU())
        self.refinement = nn.Sequential(*layers)
        
        self.projection = nn.Conv3d(width, 1, kernel_size=1)

    def forward(self, x):
        # x: [nb, 3, Pf, Pf, nt]
        # Channels: 0=Interp_Coarse, 1=Bed, 2=Bathtub
        
        # 1. Separate Feature Extraction
        water_info = x[:, [0, 2], ...] # Coarse + Bathtub
        bed_info = x[:, 1:2, ...]      # Topography
        
        f_water = self.water_net(water_info)
        f_bed = self.bed_net(bed_info)
        
        # 2. Cross-Attention: Water queries the Terrain
        # This tells the model "Where should the water go based on the land?"
        
        mixed = self.cross_attn(f_water, f_bed)
        
        # 3. Refine the mixed features
        feat = self.refinement(mixed + f_water) # Skip connection for stability
        
        # 4. Final Residual Projection
        residual = self.projection(feat)
        
        # Anchored to the Bathtub baseline
        return x[:, 2:3, ...] + residual

In [7]:
import torch

def test_flexible_magnifier():
    # --- 1. Configuration ---
    bs = 2
    f = 4
    my, mx = 10, 10
    n_t = 5
    width = 64
    
    ny, nx = my * f, mx * f # 40 x 40
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # --- 2. Initialize Model with 3 Input Channels ---
    # (Coarse, Bed, Bathtub)
    model = CrossAttentionMagnifier(width=width, num_refinement_layers=2).to(device)
    
    # --- 3. Create Augmented Batch [nb, 3, Pf, Pf, nt] ---
    # Channel 0: Interpolated Coarse
    interp_u = torch.randn(bs, 1, ny, nx, n_t).to(device)
    # Channel 1: Fine Topography
    bed_topo = torch.randn(bs, 1, ny, nx, n_t).to(device)
    # Channel 2: Bathtub Baseline (Our Physics Anchor)
    u_bathtub = torch.randn(bs, 1, ny, nx, n_t).to(device)
    
    x_input = torch.cat([interp_u, bed_topo, u_bathtub], dim=1)

    print(f"Input Shape: {x_input.shape}")

    # --- 4. Forward Pass ---
    output = model(x_input)

    print(f"Output Shape: {output.shape}")

    # --- 5. Logical Checks ---
    
    # CHECK 1: Shape Consistency
    assert output.shape == (bs, 1, ny, nx, n_t), "Output shape mismatch!"
    print("✅ Check 1: Output shape is correct.")

    # CHECK 2: Residual Anchoring Logic
    # If we set the model's final projection weights to zero, 
    # the output MUST exactly equal u_bathtub (Channel 2)
    with torch.no_grad():
        model.projection.weight.zero_()
        model.projection.bias.zero_()
        zero_res_output = model(x_input)
        
        # Calculate difference from the bathtub channel
        diff = torch.abs(zero_res_output - u_bathtub).max()
        
    print(f"✅ Check 2: Anchored to Bathtub? Max Diff: {diff.item():.2e}")
    
    if diff < 1e-6:
        print("\nSUCCESS: Model correctly treats Bathtub as the physical baseline.")
    else:
        print("\nFAILURE: Model is not correctly adding the residual to the bathtub field.")

if __name__ == "__main__":
    test_flexible_magnifier()

Input Shape: torch.Size([2, 3, 40, 40, 5])
Output Shape: torch.Size([2, 1, 40, 40, 5])
✅ Check 1: Output shape is correct.
✅ Check 2: Anchored to Bathtub? Max Diff: 0.00e+00

SUCCESS: Model correctly treats Bathtub as the physical baseline.


In [2]:
import subprocess
import os

def get_node_gpu_availability(partition="standard", account="cxs1024_cr_default"):
    # 1. Get nodes belonging to the partition and their GPU specs
    # %n: node host, %G: GRES (GPUs), %t: state (idle, alloc, mixed)
    cmd = ["sinfo", "-p", partition, "-o", "%n|%G|%t", "-h"]
    
    try:
        output = subprocess.check_output(cmd, text=True).strip().split('\n')
        
        print(f"--- GPU Availability for Partition: {partition} ---")
        print(f"{'Node Name':<15} | {'GPU Type/Total':<20} | {'Status'}")
        print("-" * 55)
        
        for line in output:
            if not line: continue
            node, gres, state = line.split('|')
            
            # Filter for nodes that actually have GPUs
            if "gpu" in gres:
                # Example gres format: gpu:a100:2(S:0-1)
                # We clean it up for readability
                gpu_info = gres.replace("(S:", " [Slots: ").replace(")", "]")
                print(f"{node:<15} | {gpu_info:<20} | {state}")
                
    except subprocess.CalledProcessError:
        print(f"Error: Could not retrieve info for partition '{partition}'.")

if __name__ == "__main__":
    # You can change these to variables or sys.argv inputs
    # MY_PARTITION = "standard"
    # MY_ACCOUNT = "cxs1024_cr_default"
    MY_PARTITION = 'mgc-mri'
    MY_ACCOUNT = 'cxs1024_mri'
    get_node_gpu_availability(MY_PARTITION, MY_ACCOUNT)

--- GPU Availability for Partition: mgc-mri ---
Node Name       | GPU Type/Total       | Status
-------------------------------------------------------
p-mc-3470       | gpu:v100s:10 [Slots: 0-39] | mix
p-mc-3471       | gpu:v100s:10 [Slots: 0-39] | mix
p-mc-3472       | gpu:v100s:10 [Slots: 0-39] | mix
p-mc-3476       | gpu:v100:4 [Slots: 0-39] | mix
p-mc-3473       | gpu:rtx_6000:3 [Slots: 0-9,20-39] | idle
p-mc-3474       | gpu:rtx_6000:3 [Slots: 0-9,20-39] | idle
p-mc-3475       | gpu:v100:4 [Slots: 0-39] | idle


In [3]:
import subprocess
import re

def get_node_gpu_availability(partition="standard", account="cxs1024_mri"):
    # %n: node host, %G: GRES (GPUs), %t: state (idle, alloc, mixed)
    # We use -N to ensure we see every node individually
    cmd = ["sinfo", "-p", partition, "-N", "-o", "%n|%G|%t", "-h"]
    
    try:
        output = subprocess.check_output(cmd, text=True).strip().split('\n')
        
        print(f"--- GPU Capacity Report: {partition} ---")
        print(f"{'Node Name':<12} | {'GPU Type':<12} | {'Total':<6} | {'Idle':<6} | {'Status'}")
        print("-" * 65)
        
        for line in output:
            if not line or "|" not in line: continue
            node, gres, state = line.split('|')
            
            if "gpu" in gres:
                # Parse GRES string like 'gpu:v100s:10' or 'gpu:a100:2(S:0-1)'
                # We extract the type and the total count
                match = re.search(r'gpu:([\w\d_]+):(\d+)', gres)
                if match:
                    gpu_type = match.group(1)
                    total_gpus = int(match.group(2))
                    
                    # Logic for Idle GPUs:
                    # 'idle'  -> All GPUs are free
                    # 'alloc' -> All GPUs are busy
                    # 'mix'   -> Some are busy (requires scontrol for exact count)
                    if state.strip() == "idle":
                        idle_gpus = total_gpus
                    elif state.strip() == "alloc":
                        idle_gpus = 0
                    else: # 'mix' or others
                        # For 'mix', we check scontrol to be 100% accurate
                        idle_gpus = get_exact_idle_count(node, total_gpus)
                    
                    print(f"{node:<12} | {gpu_type:<12} | {total_gpus:<6} | {idle_gpus:<6} | {state}")
                
    except Exception as e:
        print(f"Error: {e}")

def get_exact_idle_count(node_name, total):
    """Parses scontrol to find exactly how many GPUs are allocated on a 'mix' node."""
    try:
        cmd = ["scontrol", "show", "node", node_name]
        out = subprocess.check_output(cmd, text=True)
        # Look for AllocTRES or GresUsed
        match = re.search(r'GresUsed=gpu:(\d+)', out)
        if match:
            allocated = int(match.group(1))
            return max(0, total - allocated)
        return "Check" # Fallback if not found
    except:
        return "?"

if __name__ == "__main__":
    # Settings for your new MGC-MRI partition
    MY_PARTITION = 'mgc-mri'
    get_node_gpu_availability(MY_PARTITION)

--- GPU Capacity Report: mgc-mri ---
Node Name    | GPU Type     | Total  | Idle   | Status
-----------------------------------------------------------------
p-mc-3470    | v100s        | 10     | Check  | mix
p-mc-3471    | v100s        | 10     | Check  | mix
p-mc-3472    | v100s        | 10     | Check  | mix
p-mc-3473    | rtx_6000     | 3      | 3      | idle
p-mc-3474    | rtx_6000     | 3      | 3      | idle
p-mc-3475    | v100         | 4      | 4      | idle
p-mc-3476    | v100         | 4      | Check  | mix


In [1]:
import subprocess
import re

def get_exact_idle_count(node_name, total):
    """Deep search for GPU allocation in scontrol to avoid 'Check' status."""
    try:
        cmd = ["scontrol", "show", "node", node_name]
        out = subprocess.check_output(cmd, text=True)
        
        # Method 1: Look for Allocated TRES (The most reliable for active jobs)
        tres_match = re.search(r'AllocTRES=.*gres/gpu=(\d+)', out)
        if tres_match:
            return max(0, total - int(tres_match.group(1)))
            
        # Method 2: Look for GresUsed (Usually shows 'gpu:X')
        used_match = re.search(r'GresUsed=gpu:(\d+)', out)
        if used_match:
            return max(0, total - int(used_match.group(1)))
            
        # Method 3: If node is 'mix' but no TRES/GresUsed found, it might be 
        # because the job is starting. We assume at least 1 is used.
        return f"<{total}" 
    except:
        return "?"

def get_node_gpu_availability(partition="mgc-mri"):
    cmd = ["sinfo", "-p", partition, "-N", "-o", "%n|%G|%t", "-h"]
    try:
        output = subprocess.check_output(cmd, text=True).strip().split('\n')
        print(f"--- Updated GPU Capacity: {partition} ---")
        print(f"{'Node Name':<12} | {'GPU Type':<12} | {'Total':<6} | {'Idle':<6} | {'Status'}")
        print("-" * 65)
        
        for line in output:
            if not line or "|" not in line: continue
            node, gres, state = line.split('|')
            if "gpu" in gres:
                match = re.search(r'gpu:([\w\d_]+):(\d+)', gres)
                if match:
                    gpu_type = match.group(1)
                    total_gpus = int(match.group(2))
                    
                    if state.strip() == "idle":
                        idle_gpus = total_gpus
                    elif state.strip() == "alloc":
                        idle_gpus = 0
                    else:
                        idle_gpus = get_exact_idle_count(node, total_gpus)
                    
                    print(f"{node:<12} | {gpu_type:<12} | {total_gpus:<6} | {idle_gpus:<6} | {state}")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    get_node_gpu_availability()

--- Updated GPU Capacity: mgc-mri ---
Node Name    | GPU Type     | Total  | Idle   | Status
-----------------------------------------------------------------
p-mc-3470    | v100s        | 10     | 3      | mix
p-mc-3471    | v100s        | 10     | 4      | mix


p-mc-3472    | v100s        | 10     | 3      | mix
p-mc-3473    | rtx_6000     | 3      | 3      | idle
p-mc-3474    | rtx_6000     | 3      | 3      | idle
p-mc-3475    | v100         | 4      | 4      | idle
p-mc-3476    | v100         | 4      | 0      | mix
